In [8]:
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import requests
import re

RAW_DIR = Path("../raw")
LINK_DIR = RAW_DIR / "lok_sabha_constituencies"

LINK_DIR.mkdir(parents=True, exist_ok=True)

PAGES = {
    2024: "https://www.myneta.info/LokSabha2024/",
    2019: "https://www.myneta.info/LokSabha2019/",
    2014: "https://www.myneta.info/ls2014/",
}

EXPECTED_COUNTS = {
    2024: 546,
    2019: 559,
    2014: 574,
}

EXPECTED_BYELECTION_COUNTS = {
    2024: 3,
    2019: 16,
    2014: 31,
}

LINK_COLUMNS = [
    "year", "state_ut", "state_id", "const", "const_id", "link",
    "anchor_title", "election_date", "is_byelection", "bye_election_date",
]

print("Working directory:", Path.cwd())
print("Output directory:", LINK_DIR.resolve())

Working directory: d:\MyCoding\DA2402\DCV-Project\notebooks
Output directory: D:\MyCoding\DA2402\DCV-Project\raw\lok_sabha_constituencies


In [9]:
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36"
    )
})

retry = Retry(
    total=4,
    connect=4,
    read=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)

session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

print("HTTP session ready.")

HTTP session ready.


In [10]:
def clean_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def extract_constituencies(html, base_url, year):
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    for a in soup.find_all("a", href=True):
        if "show_candidates" not in a["href"].lower():
            continue

        container = a.find_parent(
            "div",
            class_=lambda c: c and "w3-third" in c
        )

        button = container.find("button") if container else None

        state_link = (
            container.find(
                "a",
                href=re.compile(r"action=show_constituencies")
            )
            if container else None
        )

        link = urljoin(base_url, a["href"])
        params = parse_qs(urlparse(link).query)
        const_id = params.get("constituency_id", [None])[0]

        anchor_text = clean_text(a.get_text(" ", strip=True))
        anchor_title = clean_text(a.get("title", ""))

        bye_match = re.search(
            r"\bBYE[\s-]*ELECTION\b"
            r"(?:\s+ON\s+(\d{1,2}[-/]\d{1,2}[-/]\d{4}))?",
            anchor_text,
            flags=re.I,
        )

        const = re.split(
            r"\s*:\s*BYE[\s-]*ELECTION\b",
            anchor_text,
            maxsplit=1,
            flags=re.I,
        )[0].strip()

        date_match = re.search(
            r"Date of Election\s*:?\s*"
            r"(\d{1,2}[-/]\d{1,2}[-/]\d{4})",
            anchor_title,
            flags=re.I,
        )

        state_href = state_link["href"] if state_link else None
        state_id = (
            parse_qs(urlparse(state_href).query)
            .get("state_id", [None])[0]
            if state_href else None
        )

        rows.append({
            "year": year,
            "state_ut": (
                clean_text(button.get_text(" ", strip=True))
                if button else None
            ),
            "state_id": state_id,
            "const": const,
            "const_id": str(const_id) if const_id else None,
            "link": link,
            "anchor_title": anchor_title,
            "election_date": date_match.group(1) if date_match else None,
            "is_byelection": bye_match is not None,
            "bye_election_date": (
                bye_match.group(1) if bye_match else None
            ),
        })

    df = pd.DataFrame(rows, columns=LINK_COLUMNS)

    # Remove duplicate URLs only; do not deduplicate by constituency name.
    return df.drop_duplicates(
        subset=["link"],
        keep="first",
    ).reset_index(drop=True)

In [11]:
link_dfs = {}
audit_rows = []

for year, url in PAGES.items():
    print(f"Fetching {year}...")

    response = session.get(url, timeout=(15, 45))
    response.raise_for_status()

    df = extract_constituencies(response.content, url, year)

    output_path = LINK_DIR / f"constituency_links_{year}.csv"
    df.to_csv(output_path, index=False)

    link_dfs[year] = df

    audit_rows.append({
        "year": year,
        "expected_links": EXPECTED_COUNTS[year],
        "actual_links": len(df),
        "unique_urls": df["link"].nunique(),
        "missing_const_id": int(df["const_id"].isna().sum()),
        "missing_state": int(df["state_ut"].isna().sum()),
        "missing_state_id": int(df["state_id"].isna().sum()),
        "byelection_count": int(df["is_byelection"].sum()),
        "expected_byelection_count": EXPECTED_BYELECTION_COUNTS[year],
        "http_status": response.status_code,
    })

    print(f"  Extracted {len(df)} links -> {output_path}")

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(LINK_DIR / "constituency_link_audit.csv", index=False)

print("\nFetch complete.")

Fetching 2024...
  Extracted 546 links -> ..\raw\lok_sabha_constituencies\constituency_links_2024.csv
Fetching 2019...
  Extracted 559 links -> ..\raw\lok_sabha_constituencies\constituency_links_2019.csv
Fetching 2014...
  Extracted 574 links -> ..\raw\lok_sabha_constituencies\constituency_links_2014.csv

Fetch complete.


In [12]:
validation_rows = []

for year, df in link_dfs.items():
    bye_count = int(df["is_byelection"].sum())

    checks = {
        "correct_count": len(df) == EXPECTED_COUNTS[year],
        "unique_urls": (
            df["link"].notna().all()
            and df["link"].is_unique
        ),
        "unique_const_ids": (
            df["const_id"].notna().all()
            and df["const_id"].is_unique
        ),
        "state_metadata_present": (
            df["state_ut"].notna().all()
            and df["state_id"].notna().all()
        ),
        "correct_byelection_count": (
            bye_count == EXPECTED_BYELECTION_COUNTS[year]
        ),
    }

    validation_rows.append({
        "year": year,
        "links": len(df),
        "expected": EXPECTED_COUNTS[year],
        "byelections": bye_count,
        **checks,
        "valid": all(checks.values()),
    })

validation_df = pd.DataFrame(validation_rows)
print(validation_df.to_string(index=False))

if not validation_df["valid"].all():
    raise ValueError(
        "Validation failed. Inspect the audit and link CSVs before proceeding."
    )

print("\nAll three years passed validation.")

 year  links  expected  byelections  correct_count  unique_urls  unique_const_ids  state_metadata_present  correct_byelection_count  valid
 2024    546       546            3           True         True              True                    True                      True   True
 2019    559       559           16           True         True              True                    True                      True   True
 2014    574       574           31           True         True              True                    True                      True   True

All three years passed validation.


In [13]:
all_links = pd.concat(
    [link_dfs[year] for year in [2024, 2019, 2014]],
    ignore_index=True,
)

all_path = LINK_DIR / "constituency_links_all_years.csv"
all_links.to_csv(all_path, index=False)

print("Total links:", len(all_links))
print("Unique URLs:", all_links["link"].nunique())
print("Saved:", all_path.resolve())

print("\nCounts by year:")
print(all_links.groupby("year").size())

print("\nCounts by year and state:")
print(
    all_links.groupby(["year", "state_ut"])
    .size()
    .to_string()
)

Total links: 1679
Unique URLs: 1679
Saved: D:\MyCoding\DA2402\DCV-Project\raw\lok_sabha_constituencies\constituency_links_all_years.csv

Counts by year:
year
2014    574
2019    559
2024    546
dtype: int64

Counts by year and state:
year  state_ut                                
2014  ANDAMAN & NICOBAR ISLANDS                    1
      ANDHRA PRADESH                              25
      ARUNACHAL PRADESH                            2
      ASSAM                                       15
      BIHAR                                       41
      CHANDIGARH                                   1
      CHHATTISGARH                                11
      DADRA & NAGAR HAVELI                         1
      DAMAN & DIU                                  1
      GOA                                          2
      GUJARAT                                     27
      HARYANA                                     10
      HIMACHAL PRADESH                             4
      JAMMU & KASHMIR         

We initially used Selenium to scrape constituency links from the MyNeta Lok Sabha landing pages for 2024, 2019, and 2014, but wanted to determine whether a faster approach could produce the same complete results. We therefore compared Requests-based HTML parsing against Selenium's live DOM extraction for all three years, checking the total number of links and whether both methods returned identical URL sets. Requests matched Selenium exactly in every year: 546 links for 2024, 559 for 2019, and 574 for 2014, with no URL differences. Since the landing-page links are available directly in the HTTP response, Selenium's browser-rendering overhead is unnecessary for this stage. We therefore chose Requests + BeautifulSoup to extract and save constituency metadata into year-wise CSVs and a combined CSV, with validation checks for expected link counts, unique URLs and constituency IDs, state metadata, and by-election counts. This is faster and simpler while preserving the verified link coverage; Selenium remains necessary for candidate pages, where our earlier tests showed that Requests-based HTML did not capture the complete candidate data.